In [1]:
import pandas as pd
import sqlite3

# Load data
df = pd.read_csv("bank_commercial_full.csv")

# Create subscribed column (1 = yes, 0 = no)
df['subscribed'] = (df['y'] == 'yes').astype(int)

# Save the clean version for all future notebooks
df.to_csv("bank_commercial_clean.csv", index=False)

# Load into SQL
conn = sqlite3.connect(":memory:")
df.to_sql("bank", conn, index=False, if_exists="replace")

# Shortcut function for running SQL
def sql(query):
    return pd.read_sql_query(query, conn)

# Confirm everything is ready
print("Ready ✓")
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
print(f"Subscribed column exists: {'subscribed' in df.columns}")

Ready ✓
Rows: 41,188 | Columns: 24
Subscribed column exists: True


In [2]:
sql("SELECT * FROM bank LIMIT 5")

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,branch,deposit_value,subscribed
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,Porto North,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,Aveiro,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,Coimbra,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,Faro,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,Lisbon Central,0,0


In [3]:
sql("""
    SELECT 
        branch,
        COUNT(*)                          AS total_contacts,
        SUM(subscribed)                   AS conversions,
        ROUND(AVG(subscribed) * 100, 1)   AS cvr_pct,
        SUM(deposit_value)                AS total_revenue
    FROM bank
    GROUP BY branch
    ORDER BY cvr_pct DESC
""")

,branch,total_contacts,conversions,cvr_pct,total_revenue
0,Aveiro,3294,389,11.8,4594194
1,Evora,3267,382,11.7,4490032
2,Porto North,7450,861,11.6,10356308
3,Lisbon Central,9032,1025,11.3,12432009
4,Braga,5005,566,11.3,6944295
5,Coimbra,4965,555,11.2,6669889
6,Setubal,4045,441,10.9,5190349
7,Faro,4130,421,10.2,5090774


In [4]:
sql("""
    SELECT
        CASE 
            WHEN age < 30 THEN 'Young (Under 30)'
            WHEN age BETWEEN 30 AND 45 THEN 'Mid (30-45)'
            WHEN age BETWEEN 46 AND 60 THEN 'Senior (46-60)'
            ELSE 'Mature (60+)'
        END AS age_group,
        COUNT(*)                        AS total_contacts,
        SUM(subscribed)                 AS conversions,
        ROUND(AVG(subscribed) * 100, 1) AS cvr_pct
    FROM bank
    GROUP BY age_group
    ORDER BY cvr_pct DESC
""")

,age_group,total_contacts,conversions,cvr_pct
0,Mature (60+),910,414,45.5
1,Young (Under 30),5669,922,16.3
2,Senior (46-60),10921,1044,9.6
3,Mid (30-45),23688,2260,9.5


In [5]:
sql("""
    WITH segment_stats AS (
        SELECT
            job,
            COUNT(*)                        AS total_contacts,
            SUM(subscribed)                 AS conversions,
            ROUND(AVG(subscribed) * 100, 1) AS cvr_pct,
            SUM(deposit_value)              AS revenue
        FROM bank
        GROUP BY job
    )
    SELECT *
    FROM segment_stats
    WHERE cvr_pct > 11.3
    ORDER BY cvr_pct DESC
""")

,job,total_contacts,conversions,cvr_pct,revenue
0,student,875,275,31.4,3186781
1,retired,1720,434,25.2,5280413
2,unemployed,1014,144,14.2,1687105
3,admin.,10422,1352,13.0,16289508
